# Releitura independente das evidências
Este notebook só consulta registros existentes. Não cria schemas, tabelas ou execuções.

Cole em `run_ids` a lista JSON emitida pelo notebook 02. Execute em uma nova sessão Python para comprovar a independência do estado anterior. Se reutilizar a sessão atual, a verificação continua útil, mas não comprova uma nova sessão.

Resultado esperado: SUCESSO, SUCESSO, FALHOU (falha controlada), com hashes válidos.

**Parâmetros:** execute apenas a primeira célula de código para exibir os widgets. Preencha os campos e depois execute as demais células. Use **Run all** quando os parâmetros já estiverem preenchidos.


In [ ]:
# 1. Exibir parâmetros — execute esta célula antes de preencher os campos.
dbutils.widgets.text("catalogo", spark.catalog.currentCatalog())
dbutils.widgets.text("run_ids", "[]")

print("Preencha os widgets acima e depois execute as próximas células.")


In [ ]:
from pathlib import Path
import sys
atual = Path.cwd().resolve()
raiz = next((p for p in [atual, *atual.parents]
             if (p / "src" / "medallion").is_dir() and (p / "pyproject.toml").is_file()), None)
if raiz is None:
    raise RuntimeError("Abra este notebook na pasta Git; se necessário, use %cd com a raiz do projeto.")
if str(raiz / "src") not in sys.path:
    sys.path.insert(0, str(raiz / "src"))
from medallion.config import PipelineConfig



In [ ]:
import json
from medallion.observability.audit_delta import DeltaAuditSink

ids = json.loads(dbutils.widgets.get("run_ids"))
if not isinstance(ids, list) or len(ids) != 3 or any(not isinstance(x, str) for x in ids):
    raise ValueError("Cole a lista JSON com os três run_ids do notebook 02.")
if len(set(ids)) != 3:
    raise ValueError("Informe três tentativas distintas.")
config = PipelineConfig(catalogo=dbutils.widgets.get("catalogo").strip())
sink = DeltaAuditSink(spark, config)
resultado = [sink.verificar(run_id) for run_id in ids]
assert [r["status"] for r in resultado] == ["SUCESSO", "SUCESSO", "FALHOU"]
for registro in resultado:
    print(registro)
print("RELEITURA DAS EVIDÊNCIAS: OK")

